In [1]:
# resume_fft_detector.py
import os
import random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
from tqdm import tqdm
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image, ImageFilter
import io
from sklearn.metrics import accuracy_score, roc_auc_score

# ----------------- Config (adjust if needed) -----------------
DATA_DIR = "./watermark_dataset"
CHECKPOINT_PATH = "fft_detector_ckpt_epoch_39.pth"  # or "fft_detector_ckpt_full.pth"
# CHECKPOINT_PATH = None  # or "fft_detector_ckpt_full.pth"
BATCH_SIZE = 8
NUM_WORKERS = 0
TOTAL_NUM_EPOCHS = 100          # total epochs you want to reach (resume will continue until this)
LR = 2e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
VALIDATION_SPLIT = 0.15
NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 7.5
SAVE_EVERY_EPOCHS = 1          # how often to save full checkpoint
# ------------------------------------------------------------

# ---------------- Augmentations (image-space) ----------------
# Enriched train_transforms — each optional action wrapped in RandomApply
def jpeg_compress_pil(img: Image.Image, quality: int = 85):
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality, optimize=True)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


class RandomJPEG:
    def __init__(self, p=0.5, q_range=(60, 95)):
        self.p = p
        self.q_range = q_range

    def __call__(self, img):
        if random.random() < self.p:
            q = random.randint(self.q_range[0], self.q_range[1])
            return jpeg_compress_pil(img, q)
        return img


class RandomGaussianNoise:
    def __init__(self, p=0.5, std=0.01):
        self.p = p
        self.std = std

    def __call__(self, img):
        if random.random() < self.p:
            arr = np.array(img).astype(np.float32) / 255.0
            noise = np.random.normal(0, self.std, arr.shape).astype(np.float32)
            arr = np.clip(arr + noise, 0.0, 1.0)
            img2 = Image.fromarray((arr * 255).astype(np.uint8))
            return img2
        return img

def make_train_image_augmentations(IMAGE_SIZE):
    # Compose PIL-based augmentations (randomly applied)
    aug_list = []
    # random rotation small
    aug_list.append(
        transforms.RandomApply([transforms.RandomRotation(degrees=15)], p=0.5)
    )
    # random resized crop (sometimes)
    aug_list.append(
        transforms.RandomApply(
            [transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0))], p=0.6
        )
    )
    # horizontal flip
    aug_list.append(transforms.RandomHorizontalFlip(p=0.5))
    # color jitter
    aug_list.append(
        transforms.RandomApply([transforms.ColorJitter(0.2, 0.2, 0.1, 0.05)], p=0.6)
    )
    # JPEG
    aug_list.append(RandomJPEG(p=0.3, q_range=(60, 95)))
    # RandAugment (if torchvision supports it) - wrapped
    try:
        from torchvision.transforms import RandAugment

        aug_list.append(transforms.RandomApply([RandAugment()], p=0.25))
    except Exception:
        pass
    # Gaussian blur sometimes
    aug_list.append(
        transforms.RandomApply(
            [
                lambda img: img.filter(
                    ImageFilter.GaussianBlur(radius=random.uniform(0.1, 1.8))
                )
            ],
            p=0.25,
        )
    )
    # Add gaussian pixel noise sometimes
    aug_list.append(RandomGaussianNoise(p=0.25, std=0.02))
    # brightness jitter more finely (RandomApply)
    aug_list.append(
        transforms.RandomApply([transforms.ColorJitter(brightness=(0.8, 1.2))], p=0.5)
    )

    # final: ensure image is resized to IMAGE_SIZE (if not already)
    final = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), *aug_list])
    return final

IMAGE_SIZE = 512  # might adjust to your pipeline / VAE size
IMG_AUG = make_train_image_augmentations(IMAGE_SIZE)
TEST_AUG = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    ]
)


torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ----------------- assume helpers are available from your notebook -----------------
def transform_img(image, target_size=512):
    tform = transforms.Compose(
        [
            transforms.Resize(target_size),
            transforms.CenterCrop(target_size),
            transforms.ToTensor(),
            transforms.ConvertImageDtype(torch.float32),
        ]
    )
    image = tform(image)
    return 2.0 * image - 1.0


# ---------------- Data loader that does augment -> pipe -> forward_diffusion -> FFT ----------------
class WatermarkOnTheFlyDataset(Dataset):
    """
    Loads image files and labels, applies augmentations, then runs:
      tsr_img -> pipe.get_image_latents(sample=False) -> pipe.forward_diffusion(...) -> FFT
    Returns: (fft_channels_tensor (float32, shape (2*C, H, W)), label)
    """

    def __init__(
        self,
        file_paths,
        labels,
        pipe,
        text_embeddings,
        num_inference_steps,
        guidance_scale=1.0,
        device="cpu",
        image_aug=IMG_AUG,
        image_aug_prob=0.5,
    ):
        assert len(file_paths) == len(labels)
        self.file_paths = file_paths
        self.labels = labels
        self.pipe = pipe
        self.text_embeddings = text_embeddings
        self.num_inference_steps = num_inference_steps
        self.guidance_scale = guidance_scale
        self.device = device
        self.image_aug = image_aug
        self.image_aug_prob = image_aug_prob

    def __len__(self):
        return len(self.file_paths)

    def _load_pil(self, fp):
        # accept PIL.Image, numpy array, or path
        if isinstance(fp, Image.Image):
            return fp.convert("RGB")
        if isinstance(fp, torch.Tensor):
            # convert tensor (C,H,W) to PIL
            arr = (fp.detach().cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            return Image.fromarray(arr)
        p = Path(fp)
        img = Image.open(p).convert("RGB")
        return img

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        label = int(self.labels[idx])

        pil_img = self._load_pil(path)

        # --- AUGMENT IMAGE FIRST ---

        if self.image_aug and (random.random() < self.image_aug_prob):
            img_aug = self.image_aug(pil_img)
        else:
            img_aug = TEST_AUG(pil_img)

        # convert to tensor and to device/dtype for pipe
        tsr_img = transform_img(img_aug).unsqueeze(
            0
        )  # (1,C,H,W)
        # move to correct dtype & device for the unet/vae as the user did earlier:
        target_dtype = next(self.pipe.unet.parameters()).dtype
        tsr_img = tsr_img.to(dtype=target_dtype, device=self.device)

        # --- encode to image latents ---
        with torch.no_grad():
            image_latents = self.pipe.get_image_latents(
                tsr_img, sample=False
            )  # user's helper expects (C,H,W) or (B,C,H,W)
            # ensure batch dim
            if image_latents.ndim == 3:
                image_latents = image_latents.unsqueeze(0)

            
            # --- forward/inversion -> x_T (depending on forward_diffusion implementation)
            reversed_latents = self.pipe.forward_diffusion(
                latents=image_latents,
                text_embeddings=self.text_embeddings,
                guidance_scale=1,
                num_inference_steps=self.num_inference_steps,
            )  # expect tensor shape (B,C,H,W)


            # Keep complex-safe: cast to float32 after splitting real/imag
            # Compute FFT (complex)
            vis_latent_fft = torch.fft.fftshift(
                torch.fft.fft2(reversed_latents), dim=(-1, -2)
            )  # (B,C,H,W) complex
            # we assume batch==1
            fft_b = vis_latent_fft[0]
            # convert to float channels (real, imag) as float32
            real = fft_b.real.to(dtype=torch.float32)
            imag = fft_b.imag.to(dtype=torch.float32)
            fft_ch = torch.cat([real, imag], dim=0)  # (2*C, H, W)

        return fft_ch, torch.tensor(label, dtype=torch.long)


# ---------------- helper to collect files and labels ----------------
def discover_dataset_files(data_dir: str):
    """
    Discover files in data_dir. Support:
      - subfolders 'watermarked' and 'clean' (or any two subfolders)
      - .pt files with keys
    Returns lists: file_paths, labels
    """
    p = Path(data_dir)
    if not p.exists():
        raise RuntimeError(f"{data_dir} not found")

    # case: two subfolders inside (binary classes)
    subdirs = [d for d in p.iterdir() if d.is_dir()]
    if len(subdirs) >= 2:
        # choose the first two directories as classes
        classes = sorted(subdirs)[:2]
        file_paths = []
        labels = []
        for label, cdir in enumerate(classes):
            exts = list(cdir.glob("*"))
            for f in exts:
                if f.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp", ".pt", ".pth"]:
                    file_paths.append(str(f))
                    labels.append(label)
        return file_paths, labels

    # case: many .pt files with 'fft' or 'image' and 'label'
    pts = list(p.glob("*.pt"))
    if len(pts) > 0:
        file_paths = []
        labels = []
        for f in pts:
            try:
                d = torch.load(f)
                if isinstance(d, dict) and "label" in d:
                    file_paths.append(str(f))
                    labels.append(int(d["label"]))
            except Exception:
                continue
        if len(file_paths) > 0:
            return file_paths, labels

    # fallback: collect images in folder and try to infer labels by filename (contains 'water' or 'wm')
    imgs = [
        str(f)
        for f in p.glob("*")
        if f.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp"]
    ]
    if len(imgs) > 0:
        file_paths = []
        labels = []
        for f in imgs:
            fname = os.path.basename(f).lower()
            lbl = 0
            if "water" in fname or "wm" in fname or "marked" in fname or "1_" in fname:
                lbl = 1
            file_paths.append(f)
            labels.append(lbl)
        return file_paths, labels

    raise RuntimeError(
        "Unable to discover dataset files. Please structure dataset as subfolders or .pt files with label key."
    )


# ---------------- small model helper ----------------
def make_model(in_channels):
    model = models.resnet18(pretrained=False)
    # adapt first conv
    model.conv1 = nn.Conv2d(
        in_channels,
        model.conv1.out_channels,
        kernel_size=model.conv1.kernel_size,
        stride=model.conv1.stride,
        padding=model.conv1.padding,
        bias=(model.conv1.bias is not None),
    )
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model

# ---------------- training / eval loops ----------------
def train_epoch(model, loader, opt, crit):
    model.train()
    running_loss = 0.0
    for X, y in tqdm(loader, desc="train", leave=False):
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        opt.zero_grad()
        out = model(X)
        loss = crit(out, y)
        loss.backward()
        opt.step()
        running_loss += loss.item() * X.size(0)
    return running_loss / len(loader.dataset)

def eval_model(model, loader):
    model.eval()
    trues, preds, probs = [], [], []
    running_loss = 0.0
    with torch.no_grad():
        for X, y in tqdm(loader, desc="eval", leave=False):
            X = X.to(DEVICE)
            y = y.to(DEVICE)
            out = model(X)
            loss = crit(out, y)
            p = torch.softmax(out, dim=1)[:, 1].detach().cpu().numpy()
            pred = (p > 0.5).astype(int).tolist()
            probs.extend(p.tolist())
            preds.extend(pred)
            trues.extend(y.cpu().numpy().tolist())
            running_loss += loss.item() * X.size(0)

    acc = accuracy_score(trues, preds)
    try:
        auc = roc_auc_score(trues, probs)
    except Exception:
        auc = float("nan")
    return acc, auc, (running_loss / len(loader.dataset))
# -----------------------------------------------------------------------------------

# Validate that PIPE and TEXT_EMBEDDINGS are present (or load them here)
try:
    import torch
    import diffusers
    from diffusers import DPMSolverMultistepScheduler
    from inverse_stable_diffusion import InversableStableDiffusionPipeline

    model_id = "stabilityai/stable-diffusion-2-1-base"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    scheduler = DPMSolverMultistepScheduler.from_pretrained(model_id, subfolder="scheduler")
    pipe = InversableStableDiffusionPipeline.from_pretrained(
        model_id,
        scheduler=scheduler,
        torch_dtype=torch.float16,
        revision="fp16",
        verbose=False,
    )
    diffusers.utils.logging.disable_progress_bar()
    pipe.set_progress_bar_config(disable=True)
    pipe = pipe.to(device)

    TEXT_EMBEDDINGS = pipe.get_text_embedding("")  #
    PIPE = pipe  # make sure 'pipe' is in scope
# ------------------------------------------
except NameError:
    raise RuntimeError("Please ensure `PIPE` and `TEXT_EMBEDDINGS` are available in the runtime before running resume script.")

# discover files
file_paths, labels = discover_dataset_files(DATA_DIR)
combined = list(zip(file_paths, labels))
random.shuffle(combined)
file_paths, labels = zip(*combined)
n_val = int(len(file_paths) * VALIDATION_SPLIT)
val_paths = file_paths[:n_val]
val_labels = labels[:n_val]
train_paths = file_paths[n_val:]
train_labels = labels[n_val:]

# create datasets (use same constructor as before)
train_ds = WatermarkOnTheFlyDataset(
    train_paths,
    train_labels,
    pipe=PIPE,
    text_embeddings=TEXT_EMBEDDINGS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    device=DEVICE,
)
val_ds = WatermarkOnTheFlyDataset(
    val_paths,
    val_labels,
    pipe=PIPE,
    text_embeddings=TEXT_EMBEDDINGS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    device=DEVICE,
)

val_ds_no_aug = WatermarkOnTheFlyDataset(
    val_paths,
    val_labels,
    pipe=PIPE,
    text_embeddings=TEXT_EMBEDDINGS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    device=DEVICE,
    image_aug=None,
)


# infer input channels from a sample (may be expensive, but needed to create model)
sample_fft, _ = train_ds[0]
in_ch = sample_fft.shape[0]

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
val_loader_no_aug = DataLoader(val_ds_no_aug, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)    

# build model + optimizer + criterion
model = make_model(in_ch).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
crit = nn.CrossEntropyLoss()

start_epoch = 1
# try to load checkpoint
if os.path.exists(CHECKPOINT_PATH):
    print(f"Loading checkpoint: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    # two possible styles:
    # 1) legacy single state_dict saved by torch.save(model.state_dict())
    # 2) full checkpoint dict saved with model_state_dict, optimizer_state_dict, epoch, maybe scheduler
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
        if "optimizer_state_dict" in ckpt:
            try:
                opt.load_state_dict(ckpt["optimizer_state_dict"])
            except Exception as e:
                print("Warning: couldn't fully load optimizer state:", e)
        if "epoch" in ckpt:
            start_epoch = int(ckpt["epoch"]) + 1
        print(f"Restored model and optimizer. Resuming from epoch {start_epoch}")
    else:
        # assume ckpt is a plain state_dict
        try:
            model.load_state_dict(ckpt)
            print("Loaded model.state_dict() from checkpoint (optimizer state not present).")
        except Exception as e:
            raise RuntimeError("Checkpoint format not recognized and failed to load model:", e)
else:
    print("No checkpoint found; training from scratch.")

# optional: a scheduler - if you used one earlier and saved state, restore here
# scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)
# if 'scheduler_state_dict' in ckpt: scheduler.load_state_dict(ckpt['scheduler_state_dict'])

# training loop (resume)
for epoch in range(start_epoch, TOTAL_NUM_EPOCHS + 1):
    print(f"Epoch {epoch}/{TOTAL_NUM_EPOCHS}")
    train_loss = train_epoch(model, train_loader, opt, crit)
    print(f"  train_loss: {train_loss:.4f}")

    acc, auc, val_loss = eval_model(model, val_loader)
    print(f"  val acc (aug): {acc:.4f}  val AUROC: {auc:.4f}  val_loss: {val_loss:.4f}")

    acc, auc, val_loss = eval_model(model, val_loader_no_aug)
    print(f"  val (no aug) acc: {acc:.4f}  val (no aug) AUROC: {auc:.4f}  val (no aug) loss: {val_loss:.4f}")

    # save model checkpoint (full)
    if (epoch % SAVE_EVERY_EPOCHS) == 0 or epoch == TOTAL_NUM_EPOCHS:
        ckpt_dict = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": opt.state_dict(),
            # "scheduler_state_dict": scheduler.state_dict() if 'scheduler' in locals() else None
        }
        ckpt_name = f"fft_detector_ckpt_epoch_{epoch}.pth"
        torch.save(ckpt_dict, ckpt_name)
        print(f"Saved checkpoint: {ckpt_name}")

print("Training complete.")


c:\Users\mike8\anaconda3\envs\pytorch2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\mike8\anaconda3\envs\pytorch2\Lib\site-packages\diffusers\pipelines\pipeline_loading_utils.py:333: FutureWarning: You are loading the variant fp16 from stabilityai/stable-diffusion-2-1-base via `revision='fp16'`. This behavior is deprecated and will be removed in diffusers v1. One should use `variant='fp16'` instead. However, it appears that stabilityai/stable-diffusion-2-1-base currently does not have the required variant filenames in the 'main' branch. 
 The Diffusers team and community would be very grateful if you could open an issue: https://github.com/huggingface/diffusers/issues/new with the title 'stabilityai/stable-diffusion-2-1-base is missing fp16 files' so that the correct variant file can be added.
  warnin

Loading checkpoint: fft_detector_ckpt_epoch_39.pth
Restored model and optimizer. Resuming from epoch 40
Epoch 40/100


  train_loss: 0.2579


  val acc (aug): 0.6933  val AUROC: 0.8158  val_loss: 0.5967


  val (no aug) acc: 0.8467  val (no aug) AUROC: 0.9847  val (no aug) loss: 0.3161
Saved checkpoint: fft_detector_ckpt_epoch_40.pth
Epoch 41/100


...

  val acc (aug): 0.8133  val AUROC: 0.8957  val_loss: 0.4235


  val (no aug) acc: 0.8867  val (no aug) AUROC: 0.9841  val (no aug) loss: 0.2689
Saved checkpoint: fft_detector_ckpt_epoch_99.pth
Epoch 100/100


  train_loss: 0.1816


  val acc (aug): 0.8200  val AUROC: 0.9367  val_loss: 0.3697


  val (no aug) acc: 0.8533  val (no aug) AUROC: 0.9848  val (no aug) loss: 0.3108
Saved checkpoint: fft_detector_ckpt_epoch_100.pth
Training complete.


In [2]:
ckpt

{'epoch': 39,
 'model_state_dict': OrderedDict([('conv1.weight',
               tensor([[[[ 1.7946e-02, -1.9029e-02,  7.2881e-03,  ...,  2.4405e-02,
                          -6.4626e-02,  4.0193e-02],
                         [ 2.4917e-02, -5.7290e-03, -6.7673e-02,  ..., -3.8434e-02,
                          -2.5189e-02, -2.0184e-02],
                         [-2.4854e-02,  5.2354e-02,  3.4758e-02,  ...,  1.0660e-02,
                          -3.9919e-02, -3.7748e-02],
                         ...,
                         [-2.6196e-02, -2.2840e-02,  3.0789e-02,  ..., -3.1291e-02,
                           3.9660e-02,  7.9086e-02],
                         [ 5.9247e-02, -3.2545e-02,  8.8838e-02,  ..., -2.0823e-03,
                           2.1489e-02, -5.1746e-02],
                         [-8.0486e-02,  6.8620e-02, -7.5456e-02,  ..., -4.7564e-02,
                           7.1348e-02,  8.0957e-02]],
               
                        [[ 3.0804e-02,  2.0788e-02, -2.4675e-02,  

In [3]:
val_paths

('watermark_dataset\\watermarked\\0276.png',
 'watermark_dataset\\watermarked\\0007.png',
 'watermark_dataset\\watermarked\\0395.png',
 'watermark_dataset\\watermarked\\0422.png',
 'watermark_dataset\\clean\\0033.png',
 'watermark_dataset\\clean\\0483.png',
 'watermark_dataset\\clean\\0085.png',
 'watermark_dataset\\watermarked\\0250.png',
 'watermark_dataset\\clean\\0354.png',
 'watermark_dataset\\watermarked\\0023.png',
 'watermark_dataset\\clean\\0184.png',
 'watermark_dataset\\watermarked\\0309.png',
 'watermark_dataset\\clean\\0418.png',
 'watermark_dataset\\watermarked\\0115.png',
 'watermark_dataset\\watermarked\\0182.png',
 'watermark_dataset\\watermarked\\0001.png',
 'watermark_dataset\\watermarked\\0260.png',
 'watermark_dataset\\clean\\0049.png',
 'watermark_dataset\\watermarked\\0232.png',
 'watermark_dataset\\clean\\0336.png',
 'watermark_dataset\\clean\\0450.png',
 'watermark_dataset\\watermarked\\0290.png',
 'watermark_dataset\\clean\\0350.png',
 'watermark_dataset\\clea